# LAB 08 — Tokenization, special tokens, attention mask và MLM

Tokenizer: `distilbert-base-uncased` (WordPiece, vocab 30 522, chữ thường), cùng tokenizer dùng trong `fine_tuning.ipynb`.

1. Calculation Exercise 2: "The cats are playing." → token, ID, special token, attention mask, đếm word / token / subword;
2. `tokens.keys()`, padding trong batch và attention mask;
3. subword tokenization với từ dài, từ hiếm, lỗi chính tả, tiếng Việt, emoji;
4. Masked Language Modeling: DistilBERT dự đoán "The cat [MASK] fish." và kiểm tra lại Calculation Exercise 1.

In [1]:
import os, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"; os.environ["TRANSFORMERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore")
import math
import pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
print(type(tokenizer).__name__, "| vocab size:", tokenizer.vocab_size, "| model_max_length:", tokenizer.model_max_length)
print("special tokens:", {t: tokenizer.convert_tokens_to_ids(t) for t in tokenizer.all_special_tokens})

BertTokenizer | vocab size: 30522 | model_max_length: 512
special tokens: {'[UNK]': 100, '[SEP]': 102, '[PAD]': 0, '[CLS]': 101, '[MASK]': 103}


## 1. Calculation Exercise 2 — "The cats are playing."

In [2]:
text = "The cats are playing."
tokens = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
print("keys:", list(tokens.keys()))
ids = tokens["input_ids"][0].tolist()
toks = tokenizer.convert_ids_to_tokens(ids)
special = set(tokenizer.all_special_ids)
pd.DataFrame({"position": range(len(ids)), "token": toks, "token ID": ids,
              "special?": [i in special for i in ids],
              "attention_mask": tokens["attention_mask"][0].tolist()}).set_index("position")

keys: ['input_ids', 'token_type_ids', 'attention_mask']


,token,token ID,special?,attention_mask
position,,,,
0,[CLS],101,True,1
1,the,1996,False,1
2,cats,8870,False,1
3,are,2024,False,1
4,playing,2652,False,1
5,.,1012,False,1
6,[SEP],102,True,1


In [3]:
words = text.split()
content = [t for t, i in zip(toks, ids) if i not in special]
subword = [t for t in content if t.startswith("##")]
print("words (tách theo khoảng trắng):", len(words), words)
print("tokens (cả special):          ", len(toks))
print("tokens (bỏ special):          ", len(content), content)
print("subword tokens (bắt đầu ##):  ", len(subword), subword)
print("decode lại:", tokenizer.decode(ids, skip_special_tokens=True))

words (tách theo khoảng trắng): 4 ['The', 'cats', 'are', 'playing.']
tokens (cả special):           7
tokens (bỏ special):           5 ['the', 'cats', 'are', 'playing', '.']
subword tokens (bắt đầu ##):   0 []
decode lại: the cats are playing.


Mỗi từ ở đây đều có sẵn trong vocabulary nên **không bị tách thành subword**: "cats" và "playing" là token nguyên, không thành `cat + ##s` hay `play + ##ing`. Dấu "." bị tách khỏi "playing" thành token riêng, nên 4 word thành 5 token nội dung, cộng `[CLS]` và `[SEP]` là 7 token.

## 2. Batch: padding và attention mask

Khi các câu trong batch dài khác nhau, câu ngắn được thêm `[PAD]` (ID 0) cho bằng câu dài nhất. Attention mask = 0 ở các vị trí đó.

In [4]:
batch = ["The cat eats fish.", "The cats are playing.", "This movie is great"]
enc = tokenizer(batch, padding=True, truncation=True, return_tensors="pt")
for k, v in enc.items():
    print(k, tuple(v.shape))
for i in range(len(batch)):
    print(f"{batch[i]!r:26}", tokenizer.convert_ids_to_tokens(enc["input_ids"][i]), enc["attention_mask"][i].tolist())

input_ids (3, 7)
token_type_ids (3, 7)
attention_mask (3, 7)
'The cat eats fish.'       ['[CLS]', 'the', 'cat', 'eats', 'fish', '.', '[SEP]'] [1, 1, 1, 1, 1, 1, 1]
'The cats are playing.'    ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]'] [1, 1, 1, 1, 1, 1, 1]
'This movie is great'      ['[CLS]', 'this', 'movie', 'is', 'great', '[SEP]', '[PAD]'] [1, 1, 1, 1, 1, 1, 0]


`token_type_ids` (segment ID) dùng cho cặp câu `[CLS] A [SEP] B [SEP]`. DistilBERT không dùng nó, nên model trong `pretrained_classification.py` chỉ nhận `input_ids` và `attention_mask`.

In [5]:
pair = tokenizer("The cat eats fish.", "It is hungry.")
print(tokenizer.convert_ids_to_tokens(pair["input_ids"]))
print("token_type_ids:", pair["token_type_ids"])

['[CLS]', 'the', 'cat', 'eats', 'fish', '.', '[SEP]', 'it', 'is', 'hungry', '.', '[SEP]']
token_type_ids: [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1]


**Attention mask có tác dụng thật không?** Cùng một câu, tính representation [CLS] khi đứng một mình, khi bị pad thêm 10 token mà có mask, và khi pad mà mask toàn 1.

In [6]:
from transformers import AutoModel
enc_model = AutoModel.from_pretrained("distilbert-base-uncased").eval()
one = tokenizer("The cats are playing.", return_tensors="pt")
pad_ids = torch.cat([one["input_ids"], torch.zeros(1, 10, dtype=torch.long)], 1)
good_mask = torch.cat([one["attention_mask"], torch.zeros(1, 10, dtype=torch.long)], 1)
with torch.no_grad():
    h0 = enc_model(**one).last_hidden_state[0, 0]
    h_mask = enc_model(input_ids=pad_ids, attention_mask=good_mask).last_hidden_state[0, 0]
    h_nomask = enc_model(input_ids=pad_ids, attention_mask=torch.ones_like(pad_ids)).last_hidden_state[0, 0]
print(f"‖h[CLS] (pad + mask đúng) − h[CLS] (không pad)‖   = {(h_mask - h0).norm():.2e}")
print(f"‖h[CLS] (pad, mask toàn 1) − h[CLS] (không pad)‖  = {(h_nomask - h0).norm():.2f}   (‖h‖ = {h0.norm():.2f})")

‖h[CLS] (pad + mask đúng) − h[CLS] (không pad)‖   = 6.96e-06
‖h[CLS] (pad, mask toàn 1) − h[CLS] (không pad)‖  = 6.68   (‖h‖ = 12.33)


Có mask đúng thì padding không làm đổi representation (sai số làm tròn, khoảng 1e-6). Bỏ mask thì `[PAD]` được attend như nội dung thật và vector [CLS] đổi hẳn.

## 3. Subword tokenization

In [7]:
examples = ["unbelievable", "tokenization", "playing", "unwatchable", "overacting", "cinematography",
            "Tarantino", "Schwarzenegger", "soooo baaaad", "gr8 movie", "COVID-19",
            "phim rất hay", "Phở", "😍", "naïve café"]
rows = []
for w in examples:
    t = tokenizer.tokenize(w)
    rows.append({"input": w, "tokens": " ".join(t), "#words": len(w.split()), "#tokens": len(t),
                 "#subword (##)": sum(x.startswith("##") for x in t), "[UNK]?": "[UNK]" in t})
pd.DataFrame(rows).set_index("input")

,tokens,#words,#tokens,#subword (##),[UNK]?
input,,,,,
unbelievable,unbelievable,1,1,0,False
tokenization,token ##ization,1,2,1,False
playing,playing,1,1,0,False
unwatchable,un ##watch ##able,1,3,2,False
overacting,over ##act ##ing,1,3,2,False
cinematography,cinematography,1,1,0,False
Tarantino,tara ##ntino,1,2,1,False
Schwarzenegger,schwarz ##ene ##gger,1,3,2,False
soooo baaaad,soo ##oo ba ##aa ##ad,2,5,3,False


Nhận xét:

- **Từ phổ biến** ("playing", "unbelievable", "cinematography") là một token nguyên. Ví dụ `un + believe + able` trong đề chỉ là minh họa; WordPiece của DistilBERT không tách từ này.
- **Từ ít gặp** được tách theo mảnh có nghĩa: "unwatchable" thành `un ##watch ##able`, "overacting" thành `over ##act ##ing`. Model vẫn nhận ra gốc "watch" / "act" dù chưa thấy nguyên từ.
- **Tên riêng** bị tách thành mảnh vô nghĩa ("tara ##ntino", "schwarz ##ene ##gger"). Dù vậy không có từ nào là out-of-vocabulary như LAB 05–07 (ở đó từ ngoài top 25 000 thành `<unk>`).
- **Kéo dài chữ hoặc viết tắt** ("soooo baaaad", "gr8") bị cắt vụn. Nghĩa phải ghép lại từ nhiều mảnh không mang nghĩa.
- **Tiếng Việt:** tokenizer `uncased` bỏ dấu ("rất" thành "rat", "phở" thành `ph ##o`), nên mất thông tin thanh điệu. Emoji 😍 không có trong vocab nên thành `[UNK]`, tức mất hẳn một tín hiệu sentiment.

Đây là *tokenizer limitation*: tokenizer cố định từ lúc pretrain, không thích nghi với domain hay ngôn ngữ mới.

## 4. Masked Language Modeling

In [8]:
mlm = AutoModelForMaskedLM.from_pretrained("distilbert-base-uncased").eval()
def fill_mask(sentence, candidates=(), k=8):
    enc = tokenizer(sentence, return_tensors="pt")
    pos = (enc["input_ids"][0] == tokenizer.mask_token_id).nonzero()[0, 0]
    with torch.no_grad():
        prob = torch.softmax(mlm(**enc).logits[0, pos], -1)
    top = torch.topk(prob, k)
    print(sentence)
    print("  top-k:", ", ".join(f"{tokenizer.convert_ids_to_tokens(int(i))} {p:.3f}" for p, i in zip(top.values, top.indices)))
    for c in candidates:
        p = prob[tokenizer.convert_tokens_to_ids(c)].item()
        print(f"  P({c}) = {p:.4f} → loss −log P = {-math.log(p):.3f}")
    return prob

_ = fill_mask("The cat [MASK] fish.", ["eats", "dog", "cat", "fish"])
_ = fill_mask("This movie is [MASK]. I loved every minute of it.", ["great", "terrible"])
_ = fill_mask("This movie is [MASK]. I walked out after twenty minutes.", ["great", "terrible"])

The cat [MASK] fish.
  top-k: eats 0.351, ##tail 0.073, attracts 0.065, ate 0.045, hunts 0.031, loves 0.028, eat 0.015, hates 0.014
  P(eats) = 0.3505 → loss −log P = 1.048
  P(dog) = 0.0002 → loss −log P = 8.594
  P(cat) = 0.0000 → loss −log P = 11.711
  P(fish) = 0.0001 → loss −log P = 9.820
This movie is [MASK]. I loved every minute of it.
  top-k: amazing 0.133, incredible 0.090, epic 0.053, awesome 0.047, fantastic 0.042, unbelievable 0.041, perfect 0.036, hilarious 0.028
  P(great) = 0.0178 → loss −log P = 4.031
  P(terrible) = 0.0075 → loss −log P = 4.895
This movie is [MASK]. I walked out after twenty minutes.
  top-k: boring 0.061, awesome 0.030, bad 0.022, awful 0.021, scary 0.019, amazing 0.019, terrible 0.018, hilarious 0.017
  P(great) = 0.0094 → loss −log P = 4.665
  P(terrible) = 0.0184 → loss −log P = 3.997


Model chưa từng được train cho sentiment, nhưng chỉ nhờ MLM đã:

- đoán đúng "eats" (0.35) và các động từ hợp ngữ cảnh ("hunts", "loves"), còn "dog" / "cat" / "fish" gần 0;
- đặt top-k toàn từ tích cực ("amazing", "incredible") khi phần sau là "I loved every minute", và chuyển sang "boring", "bad", "awful" khi phần sau là "I walked out". P(great) / P(terrible) đảo chiều giữa hai câu.

Đây là bằng chứng representation pretrained đã chứa thông tin sentiment. Đó là lý do frozen encoder vẫn phân loại được (Exp 2).

### Kiểm tra Calculation Exercise 1

L = −log P(y), log tự nhiên.

In [9]:
for p in [0.70, 0.95, 0.01]:
    print(f"P(eats) = {p:.2f} → L = −ln({p}) = {-math.log(p):.4f}")
# phân phối của đề: cross-entropy với one-hot "eats" chỉ lấy đúng xác suất của token đúng
probs = torch.tensor([0.10, 0.05, 0.70, 0.15])
print("F.nll_loss(log p, eats) =", torch.nn.functional.nll_loss(probs.log().unsqueeze(0), torch.tensor([2])).item())

P(eats) = 0.70 → L = −ln(0.7) = 0.3567
P(eats) = 0.95 → L = −ln(0.95) = 0.0513
P(eats) = 0.01 → L = −ln(0.01) = 4.6052
F.nll_loss(log p, eats) = 0.3566749691963196
